# Data Cleaning Pipeline

- Cleaning RAW csv file of all 4 provinces
- Denormalizing into Products, Sales, Promo Code, Promo Spend and Revenue Tables
- Merging Products and Sales across 4 provinces

# Importing Libraries

In [1]:
from pathlib import Path
import pandas as pd
import pyodbc

In [2]:
pyodbc_drivers = list(pyodbc.drivers())
print(pyodbc_drivers)

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server']


# User Functions

## Class - Dir

In [3]:
class Dir:
    def __init__(self):
        self.notebook_dir = Path.cwd()
        self.root_dir = self.notebook_dir.parent

    def data_raw_dir(self, file_name):
        return(self.root_dir / "data" / "raw" / file_name)

    def data_cleaned_dir(self, file_name):
        return(self.root_dir / "data" / "cleaned" / file_name)

## Class - List

In [4]:
class List:
    def primary_key():
        return ['global_code']

    def titlecase_columns():
        return ['description', 'brand', 'category', 'category_prov', 'color', 'country_of_origin', 'listing_type', 'status', 'varietal']

    def notnull_columns():
        return ['description', 'brand', 'category_prov', 'color', 'country_of_origin', 'varietal']
    
    def product_attributes():
        return ['province_code', 'description', 'brand', 'varietal', 'color', 'color_five', 'country_of_origin', 
                'category_prov', 'category', 'listing_type', 'size_ml_per_unit', 'units_per_case', 'status', 
                'wholesale_price_last_year', 'wholesale_price_this_year', 
                'retail_price_last_year', 'retail_price_this_year', 'retail_price_current', 
                'price_segment', 'price_segment_retail']
    
    def period_attributes():
        return ['2022P09', '2022P10', '2022P11', '2022P12', '2022P13', 
                '2023P01', '2023P02', '2023P03', '2023P04', '2023P05', '2023P06', '2023P07', '2023P08', 
                '2023P09', '2023P10', '2023P11', '2023P12', '2023P13', 
                '2024P01', '2024P02', '2024P03', '2024P04', '2024P05', '2024P06', '2024P07', '2024P08']

    def month_attributes():
        return ['2022M11', '2022M12', '2023M01', '2023M02', '2023M03', '2023M04', 
                '2023M05', '2023M06', '2023M07', '2023M08', '2023M09', '2023M10', 
                '2023M11', '2023M12', '2024M01', '2024M02', '2024M03', '2024M04', 
                '2024M05', '2024M06', '2024M07', '2024M08', '2024M09', '2024M10']

    def leading_character_removal():
        return ['>','#','-','*','«']

# Support Files and Dictionaries

## df_color_map

In [5]:
df_color_map = pd.read_csv(Dir().data_cleaned_dir('dim_wine_color.csv'))
dict_color_map = dict(zip(df_color_map.color, df_color_map.color_five))

## df_price_segments & Intervals

In [6]:
df_price_segments = pd.read_csv(Dir().data_cleaned_dir('dim_price_segments.csv'))

In [7]:
# Making Price Intervals based on Price_Segments_CSV
intervals = pd.IntervalIndex.from_arrays(
    df_price_segments.price_range_start,
    df_price_segments.price_range_end,
    closed = "both"    
)

# Ontario - Data Cleaning

## Data Ingestion from CSV file

In [8]:
# Data Ingestion from csv files
df_on = pd.read_csv(Dir().data_raw_dir('fact_sales_on.csv'))

In [9]:
df_on.shape

(17814, 48)

## Creating Global Code and Province Code

In [10]:
df_on['province_code'] = 'ON'

In [11]:
df_on['global_code'] = df_on.province_code + df_on.code.astype(str)

## Basic Cleaning

### Dropping Zero or Negetive total Sale

In [12]:
df_on['total_sale'] = df_on[List.period_attributes()].sum(axis=1)
df_on = df_on[df_on.total_sale > 0]

### Turning all CAPS data into Propercase

In [13]:
for column in List.titlecase_columns():
    df_on[column] = df_on[column].str.strip().str.title()
    df_on[column] = df_on[column].str.replace(',','', regex=True).str.replace(r'\s+', ' ', regex=True)

In [14]:
# Special Case - USA needs to be maintained in uppercase
df_on.country_of_origin = df_on.country_of_origin.replace('Usa','USA')

### Removing leading special characters

In [15]:
for char in List.leading_character_removal():
    df_on.description = df_on.description.str.lstrip(char)

### Removing Null Records

In [16]:
for column in List.notnull_columns():
    df_on = df_on[df_on[column].notnull()]

### Removing unnecessary attributes

In [17]:
df_on.drop(columns=['code', 'total_sale'], inplace=True)

## Data Transformation

### Mapping Colors and Making 'color_five' column

In [18]:
# Making dictionary out of color mapping csv
df_on['color_five'] = df_on['color'].map(dict_color_map)

### Mapping Price Segments based on Retail Price This Year

In [19]:
# Mapping Correct Price Segment based on Average Price
df_on['price_segment_retail'] = df_price_segments['price_segment'].iloc[intervals.get_indexer(df_on['retail_price_this_year'])].values

### Cheking Missing Attribute Columns as per requirement

In [20]:
for column in List.product_attributes():
    if column in list(df_on.columns):
        pass
    else:
        print(f'Missing Attribute : {column}')

Missing Attribute : wholesale_price_last_year
Missing Attribute : wholesale_price_this_year


In [21]:
df_on['wholesale_price_last_year'] = 0
df_on['wholesale_price_this_year'] = 0

In [22]:
df_on.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15987 entries, 0 to 17813
Data columns (total 53 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   description                      15987 non-null  object 
 1   brand                            15987 non-null  object 
 2   category                         15987 non-null  object 
 3   category_prov                    15987 non-null  object 
 4   color                            15987 non-null  object 
 5   country_of_origin                15987 non-null  object 
 6   listing_type                     15987 non-null  object 
 7   price_segment                    15987 non-null  object 
 8   status                           15987 non-null  object 
 9   units_per_case                   15987 non-null  int64  
 10  varietal                         15987 non-null  object 
 11  size_ml_per_unit                 15987 non-null  int64  
 12  retail_price_last_year 

### Creating df_on_products

In [23]:
df_on_products = df_on[List.primary_key() + List.product_attributes()]

In [24]:
df_on_products.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15987 entries, 0 to 17813
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   global_code                15987 non-null  object 
 1   province_code              15987 non-null  object 
 2   description                15987 non-null  object 
 3   brand                      15987 non-null  object 
 4   varietal                   15987 non-null  object 
 5   color                      15987 non-null  object 
 6   color_five                 15987 non-null  object 
 7   country_of_origin          15987 non-null  object 
 8   category_prov              15987 non-null  object 
 9   category                   15987 non-null  object 
 10  listing_type               15987 non-null  object 
 11  size_ml_per_unit           15987 non-null  int64  
 12  units_per_case             15987 non-null  int64  
 13  status                     15987 non-null  object 


### Creating df_on_sales

In [25]:
df_on_sales = df_on[List.primary_key() + List.period_attributes()]

In [26]:
# Unpivoting Sales Column
df_on_sales = df_on_sales.melt(
    List.primary_key(),
    List.period_attributes(),
    "sale_period",
    "sale_actual_cases"
)

In [27]:
df_on_sales.sale_actual_cases = df_on_sales.sale_actual_cases.round(2)

## --- Summary ---
- df_on_products
- df_on_sales

# Quebec - Data Cleaning

## Data Ingestion from CSV file

In [28]:
# Data Ingestion from csv files
df_qc = pd.read_csv(Dir().data_raw_dir('fact_sales_qb.csv'), dtype={"promos_list_this_year": str}, low_memory=False)

In [29]:
df_qc.shape

(29395, 48)

## Creating Global Code and Province Code

In [30]:
df_qc['province_code'] = 'QC'

In [31]:
df_qc['global_code'] = df_qc.province_code + df_qc.code.astype(str)

## Basic Cleaning

### Dropping Zero or Negetive total Sale

In [32]:
df_qc['total_sale'] = df_qc[List.period_attributes()].sum(axis=1)
df_qc = df_qc[df_qc.total_sale > 0]

In [33]:
df_qc.shape

(20345, 51)

### Turning all CAPS data into Propercase

In [34]:
for column in List.titlecase_columns():
    df_qc[column] = df_qc[column].str.strip().str.title()
    df_qc[column] = df_qc[column].str.replace(',','', regex=True).str.replace(r'\s+', ' ', regex=True)

In [35]:
df_qc.country_of_origin = df_qc.country_of_origin.replace('Usa','USA')

### Removing leading special characters

In [36]:
for char in List.leading_character_removal():
    df_qc.description = df_qc.description.str.lstrip(char)

### Removing Null Records

In [37]:
for column in List.notnull_columns():
    df_qc = df_qc[df_qc[column].notnull()]

In [38]:
df_qc.isnull().sum()

code                                   0
description                            0
brand                                  0
category                               4
category_prov                          0
color                                  0
country_of_origin                      0
listing_type                           0
price_segment                          0
status                                 0
units_per_case                         0
varietal                               0
size_ml_per_unit                       0
retail_price_last_year               125
retail_price_this_year                 0
retail_price_current                   0
promos_list_last_year              18044
promos_list_this_year              18034
promo_spend_last_year                  0
promo_spend_this_year                  0
2022P09                                0
2022P10                                0
2022P11                                0
2022P12                                0
2022P13         

### Removing unnecessary attributes

In [39]:
df_qc.drop(columns=['code', 'total_sale'], inplace=True)

In [40]:
df_qc.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18819 entries, 0 to 29394
Data columns (total 49 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   description                      18819 non-null  object 
 1   brand                            18819 non-null  object 
 2   category                         18815 non-null  object 
 3   category_prov                    18819 non-null  object 
 4   color                            18819 non-null  object 
 5   country_of_origin                18819 non-null  object 
 6   listing_type                     18819 non-null  object 
 7   price_segment                    18819 non-null  object 
 8   status                           18819 non-null  object 
 9   units_per_case                   18819 non-null  int64  
 10  varietal                         18819 non-null  object 
 11  size_ml_per_unit                 18819 non-null  int64  
 12  retail_price_last_year 

## Data Transformation

### Mapping Colors and Making 'color_five' column

In [41]:
# Making dictionary out of color mapping csv
df_qc['color_five'] = df_qc['color'].map(dict_color_map)

### Mapping Price Segments based on Retail Price This Year

In [42]:
df_qc['price_segment_retail'] = df_price_segments['price_segment'].iloc[intervals.get_indexer(df_qc['retail_price_this_year'])].values

### Cheking Missing Attribute Columns as per requirement

In [43]:
for column in List.product_attributes():
    if column in list(df_qc.columns):
        pass
    else:
        print(f'Missing Attribute : {column}')

Missing Attribute : wholesale_price_last_year
Missing Attribute : wholesale_price_this_year


In [44]:
df_qc['wholesale_price_last_year'] = 0
df_qc['wholesale_price_this_year'] = 0

### Creating df_qc_products

In [45]:
df_qc_products = df_qc[List.primary_key() + List.product_attributes()]

In [46]:
df_qc_products.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18819 entries, 0 to 29394
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   global_code                18819 non-null  object 
 1   province_code              18819 non-null  object 
 2   description                18819 non-null  object 
 3   brand                      18819 non-null  object 
 4   varietal                   18819 non-null  object 
 5   color                      18819 non-null  object 
 6   color_five                 18819 non-null  object 
 7   country_of_origin          18819 non-null  object 
 8   category_prov              18819 non-null  object 
 9   category                   18815 non-null  object 
 10  listing_type               18819 non-null  object 
 11  size_ml_per_unit           18819 non-null  int64  
 12  units_per_case             18819 non-null  int64  
 13  status                     18819 non-null  object 


### Creating df_qc_sales

In [47]:
df_qc_sales = df_qc[List.primary_key() + List.period_attributes()]

In [48]:
# Unpivoting Sales Column
df_qc_sales = df_qc_sales.melt(
    List.primary_key(),
    List.period_attributes(),
    "sale_period",
    "sale_actual_cases"
)

In [49]:
df_qc_sales.sale_actual_cases = df_qc_sales.sale_actual_cases.round(2)

In [50]:
df_qc_sales

,global_code,sale_period,sale_actual_cases
0,QC10327701,2022P09,6911.08
1,QC710780,2022P09,6221.58
2,QC964601,2022P09,5630.08
3,QC978874,2022P09,5777.92
4,QC563338,2022P09,2347.83
...,...,...,...
489289,QC15198793,2024P08,-0.08
489290,QC12314429,2024P08,0.00
489291,QC15038546,2024P08,0.00
489292,QC14645740,2024P08,1.42


## --- Summary ---
- df_qc_products
- df_qc_sales

# Alberta - Data Cleaning

## Data Ingestion from CSV file

In [51]:
# Data Ingestion from csv files
df_ab = pd.read_csv(Dir().data_raw_dir('fact_sales_ab.csv'), low_memory=False)

In [52]:
df_ab.shape

(14729, 46)

## Creating Global Code and Province Code

In [53]:
df_ab['province_code'] = 'AB'

In [54]:
df_ab['global_code'] = df_ab.province_code + df_ab.code.astype(str)

In [57]:
df_ab.shape

(14729, 48)

## Basic Cleaning

### Dropping Zero or Negetive total Sale

In [59]:
df_ab['total_sale'] = df_ab[List.month_attributes()].sum(axis=1)
df_ab = df_ab[df_ab.total_sale > 0]

In [60]:
df_ab.shape

(11390, 49)

### Turning all CAPS data into Propercase

In [62]:
for column in List.titlecase_columns():
    df_ab[column] = df_ab[column].str.strip().str.title()
    df_ab[column] = df_ab[column].str.replace(',','', regex=True).str.replace(r'\s+', ' ', regex=True)

### Removing leading special characters

In [63]:
for char in List.leading_character_removal():
    df_ab.description = df_ab.description.str.lstrip(char)

### Removing Null Records

In [64]:
for column in List.notnull_columns():
    df_ab = df_ab[df_ab[column].notnull()]

### Removing unnecessary attributes

In [65]:
df_ab.drop(columns=['code', 'total_sale'], inplace=True)

In [66]:
df_ab.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10582 entries, 0 to 14728
Data columns (total 47 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   description                      10582 non-null  object 
 1   brand                            10582 non-null  object 
 2   category                         10568 non-null  object 
 3   category_prov                    10582 non-null  object 
 4   color                            10582 non-null  object 
 5   country_of_origin                10582 non-null  object 
 6   listing_type                     10582 non-null  object 
 7   price_segment                    10582 non-null  object 
 8   status                           10582 non-null  object 
 9   units_per_case                   10582 non-null  int64  
 10  varietal                         10582 non-null  object 
 11  size_ml_per_unit                 10582 non-null  int64  
 12  retail_price_last_year 

## Data Transformation

### Mapping Colors and Making 'color_five' column

In [67]:
# Making dictionary out of color mapping csv
df_ab['color_five'] = df_ab['color'].map(dict_color_map)

### Mapping Price Segments based on Retail Price This Year

In [68]:
df_ab['price_segment_retail'] = df_price_segments['price_segment'].iloc[intervals.get_indexer(df_ab['retail_price_this_year'])].values

### Cheking Missing Attribute Columns as per requirement

In [69]:
for column in List.product_attributes():
    if column in list(df_ab.columns):
        pass
    else:
        print(f'Missing Attribute : {column}')

Missing Attribute : wholesale_price_last_year
Missing Attribute : wholesale_price_this_year


In [70]:
df_ab['wholesale_price_last_year'] = 0
df_ab['wholesale_price_this_year'] = 0

### Creating df_ab_products

In [71]:
df_ab_products = df_ab[List.primary_key() + List.product_attributes()]

In [73]:
df_ab_products.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10582 entries, 0 to 14728
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   global_code                10582 non-null  object 
 1   province_code              10582 non-null  object 
 2   description                10582 non-null  object 
 3   brand                      10582 non-null  object 
 4   varietal                   10582 non-null  object 
 5   color                      10582 non-null  object 
 6   color_five                 10582 non-null  object 
 7   country_of_origin          10582 non-null  object 
 8   category_prov              10582 non-null  object 
 9   category                   10568 non-null  object 
 10  listing_type               10582 non-null  object 
 11  size_ml_per_unit           10582 non-null  int64  
 12  units_per_case             10582 non-null  int64  
 13  status                     10582 non-null  object 


### Creating df_ab_sales

In [74]:
df_ab_sales = df_ab[List.primary_key() + List.month_attributes()]

In [75]:
# Unpivoting Sales Column
df_ab_sales = df_ab_sales.melt(
    List.primary_key(),
    List.month_attributes(),
    "sale_period",
    "sale_actual_cases"
)

In [76]:
df_ab_sales.sale_actual_cases = df_ab_sales.sale_actual_cases.round(2)

## --- Summary ---
- df_qc_products
- df_qc_sales

# British Columbia - Data Cleaning

## Data Ingestion from CSV file

In [77]:
df_bc = pd.read_csv(Dir().data_raw_dir('fact_sales_bc.csv'), low_memory=False)

In [78]:
df_bc.shape

(37889, 44)

## Creating Global Code and Province Code

In [79]:
df_bc['province_code'] = 'BC'

In [80]:
df_bc['global_code'] = df_bc.province_code + df_bc.code.astype(str)

## Basic Cleaning

### Dropping Zero or Negetive total Sale

In [81]:
df_bc['total_sale'] = df_bc[List.month_attributes()].sum(axis=1)
df_bc = df_bc[df_bc.total_sale > 0]

In [83]:
df_bc.shape

(35737, 47)

### Turning all CAPS data into Propercase

In [84]:
for column in List.titlecase_columns():
    df_bc[column] = df_bc[column].str.strip().str.title()
    df_bc[column] = df_bc[column].str.replace(',','', regex=True).str.replace(r'\s+', ' ', regex=True)

### Removing leading special characters

In [85]:
for char in List.leading_character_removal():
    df_bc.description = df_bc.description.str.lstrip(char)

### Removing Null Records

In [86]:
for column in List.notnull_columns():
    df_bc = df_bc[df_bc[column].notnull()]

### Removing unnecessary attributes

In [87]:
df_bc.drop(columns=['code', 'total_sale'], inplace=True)

## Data Transformation

### Mapping Colors and Making 'color_five' column

In [88]:
# Making dictionary out of color mapping csv
df_bc['color_five'] = df_bc['color'].map(dict_color_map)

### Mapping Price Segments based on Retail Price This Year

In [89]:
df_bc['price_segment_retail'] = df_price_segments['price_segment'].iloc[intervals.get_indexer(df_bc['retail_price_this_year'])].values

### Cheking Missing Attribute Columns as per requirement

In [90]:
for column in List.product_attributes():
    if column in list(df_ab.columns):
        pass
    else:
        print(f'Missing Attribute : {column}')

### Creating df_qc_products

In [91]:
df_bc_products = df_bc[List.primary_key() + List.product_attributes()]

### Creating df_qc_sales

In [92]:
df_bc_sales = df_bc[List.primary_key() + List.month_attributes()]

In [93]:
# Unpivoting Sales Column
df_bc_sales = df_bc_sales.melt(
    List.primary_key(),
    List.month_attributes(),
    "sale_period",
    "sale_actual_cases"
)

In [94]:
df_bc_sales.sale_actual_cases = df_bc_sales.sale_actual_cases.round(2)

## --- Summary ---
- df_qc_products
- df_qc_sales

# Ingesting Product and Sales into SQL

In [128]:
df_products = pd.concat([df_on_products, df_qc_products, df_ab_products, df_bc_products], ignore_index=True)

In [129]:
numeric_cols = [
    'wholesale_price_last_year', 'wholesale_price_this_year',
    'retail_price_last_year', 'retail_price_this_year', 'retail_price_current'
]

df_products[numeric_cols] = (
    df_products[numeric_cols]
      .apply(pd.to_numeric, errors='coerce')  # convert to numeric, invalid → NaN
      .fillna(0)                              # replace NaN with 0
      .round(2)                               # round to 2 decimal places
)

In [132]:
df_sales = pd.concat([df_on_sales, df_qc_sales, df_ab_sales, df_bc_sales], ignore_index=True)

## Connecting with SQL Database

In [131]:
df_products.to_csv(Dir().data_cleaned_dir('fact_products.csv'),encoding='utf-8', index=False)

In [133]:
df_sales.to_csv(Dir().data_cleaned_dir('fact_sales.csv'),encoding='utf-8', index=False)

In [134]:
df_sales

,global_code,sale_period,sale_actual_cases
0,ON656561,2022P09,7168.67
1,ON461053,2022P09,12264.08
2,ON35386,2022P09,6203.42
3,ON647644,2022P09,5866.17
4,ON451336,2022P09,6856.42
...,...,...,...
1951351,BC263585,2024M10,0.00
1951352,BC171929,2024M10,0.00
1951353,BC42063,2024M10,0.00
1951354,BC421616,2024M10,0.00
